# 05 - Implantação e Predição (Deployment)

Neste notebook, demonstramos como utilizar o modelo treinado de Pegada de Carbono para realizar predições em tempo real. O modelo foi salvo em formato `.joblib` e contém todo o pipeline de pré-processamento necessário.

In [ ]:
import pandas as pd
import joblib
import os

# Carregar o modelo
model_path = os.path.join('..', 'models', 'carbon_footprint_rf_v1.joblib')
model = joblib.load(model_path)

print("✅ Modelo carregado com sucesso!")

## 1. Função de Predição Customizada

Criamos uma função auxiliar que recebe os dados de entrada, os formata conforme o esperado pelo modelo e retorna o valor da emissão de CO2.

In [ ]:
def estimate_co2(energy_kwh, state, usage_type, energy_source, month):
    # Determinar a estação com base no mês
    def get_season(m):
        if m in [12, 1, 2]: return 'Verao'
        if m in [3, 4, 5]: return 'Outono'
        if m in [6, 7, 8]: return 'Inverno'
        return 'Primavera'
    
    season = get_season(month)
    
    # Criar DataFrame de entrada
    input_df = pd.DataFrame([{
        'energy_kwh': energy_kwh,
        'state': state,
        'usage_type': usage_type,
        'energy_source': energy_source,
        'month': month,
        'season': season
    }])
    
    # Realizar a predição
    prediction = model.predict(input_df)[0]
    return prediction

print("Função 'estimate_co2' pronta para uso.")

## 2. Exemplos de Uso

Vamos testar o modelo com alguns cenários hipotéticos.

In [ ]:
# Cenário A: Indústria em SP usando fonte Térmica (Gasolina/Óleo) em Janeiro
co2_a = estimate_co2(energy_kwh=1000, state='SP', usage_type='industrial', energy_source='thermal', month=1)

# Cenário B: Residência na BA usando fonte Renovável (Solar/Eólica) em Agosto
co2_b = estimate_co2(energy_kwh=1000, state='BA', usage_type='residencial', energy_source='solar', month=8)

print(f"Cenário A: {co2_a:.2f} kg CO2")
print(f"Cenário B: {co2_b:.2f} kg CO2")
print(f"Diferença: {co2_a - co2_b:.2f} kg CO2")

## 3. Conclusão

O modelo está pronto para ser integrado a sistemas de monitoramento ESG, permitindo o cálculo automatizado de emissões baseando-se apenas em dados de consumo e contexto geográfico.